# STAIR-RAM v5 — Kaggle training notebook

Pipeline: **cấu hình → môi trường → dữ liệu → preflight → train (Stage A + B) → biểu đồ → báo cáo → đóng gói**.

**Stage A** giữ nguyên MI/FSC/BSC/BPR và chọn teacher bằng validation NDCG@20 (500 epochs).  
**Stage B** load checkpoint được chọn, đóng băng toàn bộ backbone, học residual đa modality (tối đa 50/100 epochs).  
Notebook gọi mã nguồn thật; không chứa triển khai mô hình thứ hai.

Mặc định **pilot**: Stage A vẫn **500 epochs**, Stage B tối đa **50 epochs**, không mở test.  
Chuyển `MODE='confirm'` sau khi khóa cấu hình để xuất test tại checkpoint được chọn.  
Không có cam kết tăng trên 6%; dùng paired controls/seeds để kiểm chứng.

Upload/push đầy đủ mã v5 trước khi chạy. Bật GPU và Internet nếu cần clone/cài dependencies.  
Attach cả ba dataset nếu dùng Run All.


## 1. Bảng điều khiển

Mặc định chạy cả ba dataset, seed 1, arm mm-ss (primary multimodal residual).  
Các cell train dùng `selected_only=True`; gọi `train_dataset('electronics')` để chạy riêng ngoài danh sách.  
Teacher chỉ train một lần; các arm khác dùng chung teacher cùng dataset/seed.  
Để chạy thêm control, thêm arm vào ARMS: `'mm-bpr'`, `'id-ss'`, `'baseline-ss'`, `'baseline-bpr'`.


In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, shutil, subprocess, importlib.metadata, zipfile

ON_KAGGLE = Path('/kaggle/working').exists()
WORK      = Path('/kaggle/working') if ON_KAGGLE else Path.cwd()
REPO      = WORK / 'STAIR-Enhanced' if ON_KAGGLE else Path.cwd()
REPO_URL  = 'https://github.com/ThanhChuong12/STAIR-Enhanced.git'
EXPECTED_COMMIT    = None   # Pin to a specific commit SHA for reproducibility.
INSTALL_DEPENDENCIES = True
DEVICE    = '0' if ON_KAGGLE else 'cpu'   # 'cpu' for local smoke/debug only.

MODE             = 'pilot'        # 'pilot': no test | 'confirm': selected test once
DATASETS_TO_RUN  = ['baby', 'sports', 'electronics']
DATASET_SOURCES  = {'baby': None, 'sports': None, 'electronics': None}
SEEDS            = [1]            # Confirmation: [1, 2, 3, 4, 5] at minimum.
ARMS             = ['mm-ss']      # Controls: mm-bpr, id-ss, baseline-ss, baseline-bpr
HEAD_EPOCHS      = 50 if MODE == 'pilot' else 100
MAKE_ARCHIVE     = True

DATASET_NAMES = {
    'baby':        'Amazon2014Baby_550_MMRec',
    'sports':      'Amazon2014Sports_550_MMRec',
    'electronics': 'Amazon2014Electronics_550_MMRec',
}
REQUIRED_FILES = ('train.txt', 'valid.txt', 'test.txt',
                  'textual_modality.pkl', 'visual_modality.pkl')
VALID_ARMS = {'mm-ss', 'mm-bpr', 'id-ss', 'baseline-ss', 'baseline-bpr'}

EXPERIMENT_ID  = time.strftime('v5_%Y%m%d_%H%M%S')
ARTIFACT_ROOT  = WORK / 'stair4_v5_runs' / EXPERIMENT_ID
REPORT_DIR     = ARTIFACT_ROOT / 'reports'
DATA_ROOT      = WORK / 'stair4_v5_data'
CACHE_ROOT     = WORK / 'stair4_v5_cache'
INPUT_ROOT     = Path('/kaggle/input')

assert MODE in {'pilot', 'confirm'}, f'Unknown MODE: {MODE}'
assert DATASETS_TO_RUN and set(DATASETS_TO_RUN) <= set(DATASET_NAMES)
assert ARMS and set(ARMS) <= VALID_ARMS
assert len(set(SEEDS)) == len(SEEDS) and all(isinstance(s, int) and s >= 0 for s in SEEDS)

for directory in (ARTIFACT_ROOT, REPORT_DIR, CACHE_ROOT, DATA_ROOT / 'Processed'):
    directory.mkdir(parents=True, exist_ok=True)

ENV = dict(os.environ, PYTHONUTF8='1', PYTHONUNBUFFERED='1',
           OMP_NUM_THREADS='2', MKL_NUM_THREADS='2',
           CUBLAS_WORKSPACE_CONFIG=':4096:8', MPLBACKEND='Agg')

print({'mode': MODE, 'datasets': DATASETS_TO_RUN, 'seeds': SEEDS,
       'arms': ARMS, 'stage_a_epochs': 500, 'head_epochs': HEAD_EPOCHS,
       'output': str(ARTIFACT_ROOT)})


## 2. Môi trường và snapshot mã nguồn

Clone khi repo chưa tồn tại; giữ nguyên checkout có sẵn.  
Pip bị ràng buộc theo Torch đang cài để giữ CUDA wheel.  
Imports được kiểm tra trong subprocess mới; không xoá module kernel.


In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)

required_code = [
    'main_stair4_v5.py', 'models/stair4_v5.py', 'models/stair4_v5_features.py',
    'models/stair4_v5_heads.py', 'models/stair4_v5_sampling.py',
    'models/stair4_v5_objectives.py', 'models/stair4_v5_utils.py',
    'scripts/analyze_stair4_v5.py', 'requirements-stair4-v5.txt',
]
missing = [p for p in required_code if not (REPO / p).is_file()]
if missing:
    raise FileNotFoundError(f'Upload or push the missing v5 files first: {missing}')

try:
    GIT_SHA = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    GIT_STATUS = subprocess.check_output(
        ['git', 'status', '--short'], cwd=REPO, text=True)
except (subprocess.CalledProcessError, FileNotFoundError):
    GIT_SHA, GIT_STATUS = 'unavailable', 'Uploaded source without Git metadata'

if EXPECTED_COMMIT and GIT_SHA != EXPECTED_COMMIT:
    raise RuntimeError(f'Expected commit {EXPECTED_COMMIT}, found {GIT_SHA}')
print('Commit:', GIT_SHA)
print(GIT_STATUS)

torch_version_before = importlib.metadata.version('torch')
if INSTALL_DEPENDENCIES:
    constraint = ARTIFACT_ROOT / 'torch-constraint.txt'
    constraint.write_text(f'torch=={torch_version_before}\n', encoding='utf-8')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r',
         str(REPO / 'requirements-stair4-v5.txt'), '-c', str(constraint),
         'matplotlib', 'pandas'],
        cwd=REPO, env=ENV, check=True)
if importlib.metadata.version('torch') != torch_version_before:
    raise RuntimeError('The installed PyTorch version changed after pip install')

# Runtime probe — runs in a fresh interpreter to avoid stale imports.
probe = """
import json, platform, importlib.metadata, torch
import models.freerec_compat
from models.stair4_v5 import STAIR4V5
import main_stair4_v5
def _safe_ver(n):
    try: return importlib.metadata.version(n)
    except Exception: return 'not-installed'
r = {'python': platform.python_version(), 'cuda': torch.cuda.is_available(),
     'packages': {n: _safe_ver(n) for n in
                  ('torch', 'freerec', 'torchdata', 'torch-geometric', 'numpy', 'scipy', 'numba')}}
if r['cuda']:
    r['gpu'] = torch.cuda.get_device_name(0)
    r['vram_gib'] = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)
print('V5_RUNTIME_JSON=' + json.dumps(r))
"""
checked = subprocess.run(
    [sys.executable, '-c', probe], cwd=REPO, env=ENV,
    check=True, capture_output=True, text=True, encoding='utf-8')
print(checked.stderr)
RUNTIME = json.loads(
    next(x for x in checked.stdout.splitlines() if x.startswith('V5_RUNTIME_JSON='
    )).split('=', 1)[1])
if DEVICE != 'cpu' and not RUNTIME['cuda']:
    raise RuntimeError('Enable a Kaggle GPU accelerator or explicitly set DEVICE="cpu"')
print(json.dumps(RUNTIME, indent=2))
(ARTIFACT_ROOT / 'runtime.json').write_text(json.dumps(RUNTIME, indent=2), encoding='utf-8')
(ARTIFACT_ROOT / 'pip-freeze.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], env=ENV, text=True),
    encoding='utf-8')


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def code_fingerprints():
    paths = [
        REPO / 'main.py', REPO / 'main_stair4_v4.py', REPO / 'main_stair4_v5.py',
        REPO / 'requirements-stair4-v4.txt', REPO / 'requirements-stair4-v5.txt',
        REPO / 'scripts/analyze_stair4_v5.py',
        *sorted((REPO / 'models').glob('*.py')),
        *sorted((REPO / 'optimizers').glob('*.py')),
        *sorted((REPO / 'configs').glob('dataset_stair4_v5_*.yaml')),
        *sorted((REPO / 'configs').glob('Amazon2014*_550_MMRec.yaml')),
        *sorted((REPO / 'tests').glob('test_stair4_v5*.py')),
        REPO / 'tests/test_stair4_v4.py',
    ]
    paths = [p for p in paths if p.is_file()]
    return {p.relative_to(REPO).as_posix(): file_sha256(p) for p in paths}

SOURCE_HASHES = code_fingerprints()
source_manifest = ARTIFACT_ROOT / 'source_manifest.json'
if source_manifest.exists():
    previous = json.loads(source_manifest.read_text(encoding='utf-8'))
    if previous.get('files') != SOURCE_HASHES:
        raise RuntimeError('Source changed within this experiment. Use a new EXPERIMENT_ID.')
source_manifest.write_text(
    json.dumps({'git_sha': GIT_SHA, 'git_status': GIT_STATUS,
                'files': SOURCE_HASHES}, indent=2), encoding='utf-8')
for relative in SOURCE_HASHES:
    dst = ARTIFACT_ROOT / 'source' / relative
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / relative, dst)
print('Recorded source files:', len(SOURCE_HASHES))


## 3. Dò và liên kết dữ liệu

Chỉ nhận thư mục có đủ năm file FreeRec đã xử lý và feature rows tương ứng.  
Không tự split/reindex hoặc chuyển định dạng. Nếu có nhiều bản cùng tên, đặt `DATASET_SOURCES` chính xác.  
Đọc SHA-256 theo block, sau đó liên kết từng file vào thư mục writable.


In [ ]:
def discover_dataset(key, explicit=None, optional=False):
    """Find the dataset directory; require exactly one match."""
    if explicit is not None:
        p = Path(explicit).resolve()
        if not p.is_dir() or not all((p / n).is_file() for n in REQUIRED_FILES):
            raise FileNotFoundError(f'Incomplete dataset at explicit path: {p}')
        return p
    aliases = {'baby': ('baby',), 'sports': ('sport',), 'electronics': ('electronic',)}[key]
    search_roots = [INPUT_ROOT, REPO / 'data']
    candidates = set()
    for root in search_roots:
        if not root.is_dir():
            continue
        for r, dirs, files in os.walk(root, followlinks=False):
            p = Path(r)
            if (set(REQUIRED_FILES).issubset(files)
                    and any(a in str(p).lower() for a in aliases)):
                candidates.add(p.resolve())
    if len(candidates) > 1:
        raise RuntimeError(
            f'{key}: ambiguous — found {sorted(map(str, candidates))}. '
            f'Set DATASET_SOURCES[{key!r}] explicitly.')
    if candidates:
        return candidates.pop()
    if optional:
        return None
    raise FileNotFoundError(
        f'{key}: no complete dataset found. Attach it and set DATASET_SOURCES[{key!r}].')


def stage_dataset(source, key):
    """Link dataset files into a writable staging directory keyed by fingerprint."""
    source = Path(source).resolve()
    names = list(REQUIRED_FILES)
    names += [p.name for p in sorted(source.iterdir())
              if p.is_file() and p.name not in names
              and any(x in p.name.lower() for x in ('mapping', 'id2', '2id', 'config'))]
    fingerprint = {
        name: {'bytes': (source / name).stat().st_size,
               'sha256': file_sha256(source / name)}
        for name in names
    }
    key_hash = hashlib.sha256(
        json.dumps(fingerprint, sort_keys=True).encode()).hexdigest()[:16]
    root = DATA_ROOT / key_hash
    destination = root / 'Processed' / DATASET_NAMES[key]
    destination.mkdir(parents=True, exist_ok=True)
    for name in names:
        src, dst = source / name, destination / name
        if dst.exists() or dst.is_symlink():
            if dst.resolve() == src.resolve():
                continue
            if dst.is_file() and file_sha256(dst) == fingerprint[name]['sha256']:
                continue
            raise RuntimeError(f'Existing staged file differs: {dst}')
        try:
            dst.symlink_to(src)
        except OSError:
            shutil.copy2(src, dst)
    return {'source': str(source), 'root': str(root),
            'processed': str(destination), 'fingerprint': key_hash,
            'files': fingerprint}


PREPARED = {}
for key in DATASETS_TO_RUN:
    src = discover_dataset(key, DATASET_SOURCES.get(key), optional=True)
    if src is not None:
        PREPARED[key] = stage_dataset(src, key)
        print(key, '->', PREPARED[key]['processed'])
    else:
        print(f"Notice: '{key}' not found during initial scan; staged on-demand when training.")
if PREPARED:
    (ARTIFACT_ROOT / 'data_manifest.json').write_text(
        json.dumps(PREPARED, indent=2), encoding='utf-8')


## 4. Preflight — kiểm tra môi trường và cấu hình

Chạy toàn bộ v5 test suite. Mọi test failure làm pipeline dừng trước khi tốn GPU.  
Deprecation warning TorchData/CSR được giữ lại — không phải lỗi.


In [ ]:
tests = sorted(str(p.relative_to(REPO)) for p in (REPO / 'tests').glob('test_stair4_v5*.py'))
if not tests:
    raise FileNotFoundError('Upload the v5 test suite first (tests/test_stair4_v5*.py)')

preflight = subprocess.run(
    [sys.executable, '-m', 'pytest', *tests, '-q', '-p', 'no:cacheprovider',
     '--basetemp', str(ARTIFACT_ROOT / 'preflight-tmp')],
    cwd=REPO, env=ENV, capture_output=True, text=True, encoding='utf-8')
(ARTIFACT_ROOT / 'preflight.log').write_text(
    preflight.stdout + preflight.stderr, encoding='utf-8')
print(preflight.stdout)
print(preflight.stderr)
preflight.check_returncode()

# Verify CLI is reachable.
subprocess.run([sys.executable, 'main_stair4_v5.py', '--help'],
               cwd=REPO, env=ENV, check=True)
print('Preflight passed.')


## 5. Hàm helper training

Mỗi run có thư mục riêng với `command.json`, `process.log`, và các artifact v5 chuẩn.  
Teacher (Stage A) chỉ train một lần; tất cả arm dùng chung teacher cùng dataset/seed.  
Interrupt terminate subprocess; plotting failure không huỷ artifact training đã lưu.


In [ ]:
def read_jsonl(path):
    """Read all valid JSONL records from path; return [] if file does not exist."""
    path = Path(path)
    if not path.is_file():
        return []
    rows = []
    for lineno, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Malformed JSONL at {path}:{lineno}') from exc
    return rows


def launch(command, folder):
    """Run a subprocess, streaming stdout/stderr to console and process.log."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'command.json').write_text(json.dumps(command, indent=2), encoding='utf-8')
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = None
    try:
        with (folder / 'process.log').open('a', encoding='utf-8') as log:
            process = subprocess.Popen(
                command, cwd=REPO, env=ENV,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, encoding='utf-8', errors='replace', bufsize=1)
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            rc = process.wait()
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    if rc:
        raise RuntimeError(
            f'Command exited with code {rc}; inspect {folder / "process.log"}')


def manifest_completed(folder):
    """Return True if the stage at folder has a completed manifest."""
    file = Path(folder) / 'run_manifest.json'
    if not file.is_file():
        return False
    m = json.loads(file.read_text(encoding='utf-8'))
    if m.get('status') != 'completed':
        raise RuntimeError(
            f'Incomplete run at {folder}; resume explicitly or choose a new EXPERIMENT_ID')
    if m.get('test_disclosed') != (MODE == 'confirm'):
        raise RuntimeError('Do not mix pilot and confirm artifacts; use a new EXPERIMENT_ID')
    return True


def ensure_prepared(key):
    """Stage dataset on demand when a training cell requests a dataset not yet staged."""
    if key not in PREPARED:
        src = discover_dataset(key, DATASET_SOURCES.get(key), optional=False)
        PREPARED[key] = stage_dataset(src, key)
        manifest = ARTIFACT_ROOT / 'data_manifest.json'
        manifest.write_text(json.dumps(PREPARED, indent=2), encoding='utf-8')
        print(f'Staged {key} -> {PREPARED[key]["processed"]}')
    return PREPARED[key]


def base_command(key, seed):
    """Build the base CLI command for main_stair4_v5.py."""
    info = ensure_prepared(key)
    cmd = [
        sys.executable, 'main_stair4_v5.py',
        '--config',            f'configs/dataset_stair4_v5_{key}.yaml',
        '--root',              info['root'],
        '--device',            DEVICE,
        '--seed',              str(seed),
        '--epochs',            '500',
        '--head-epochs',       str(HEAD_EPOCHS),
        '--graph-cache-dir',   str(CACHE_ROOT / 'graph'),
        '--feature-cache-dir', str(CACHE_ROOT / 'features'),
    ]
    if MODE == 'pilot':
        cmd.append('--no-final-test')
    return cmd


def train_dataset(key, *, selected_only=False):
    """Train teacher and all arms for the given dataset.

    Calling train_dataset('electronics') explicitly always runs that dataset
    even if DATASETS_TO_RUN does not include it.
    Plotting failure does not convert a successful training artifact into a failure.
    """
    if selected_only and key not in DATASETS_TO_RUN:
        print(f'Skipped by Run All selection: {key}. Call train_dataset({key!r}) to force.')
        return
    for seed in SEEDS:
        seed_dir = ARTIFACT_ROOT / key / f'seed{seed}'
        teacher = seed_dir / 'teacher'
        # Stage A: train teacher once per dataset/seed.
        if not manifest_completed(teacher / 'stage_a'):
            launch(base_command(key, seed) + ['--stage', 'a', '--run-dir', str(teacher)],
                   teacher)
        # Stage B: train each arm using the shared teacher.
        for arm in ARMS:
            run = seed_dir / arm
            if not manifest_completed(run / 'stage_b'):
                launch(
                    base_command(key, seed) + [
                        '--stage',       'b',
                        '--arm',         arm,
                        '--teacher-dir', str(teacher / 'stage_a'),
                        '--run-dir',     str(run),
                    ],
                    run)
    try:
        _plot_after_training(key)
    except Exception as exc:
        print(f'Plotting failed after successful training ({type(exc).__name__}: {exc}). '
              'Re-run the dedicated plot cells to retry without retraining.')


def resume_stage_b(key, seed, arm):
    """Resume an interrupted Stage B run."""
    run = ARTIFACT_ROOT / key / f'seed{seed}' / arm
    command = json.loads((run / 'command.json').read_text(encoding='utf-8'))
    if '--resume' not in command:
        command.append('--resume')
    launch(command, run)

print('Runner functions defined.')


## 6. Hàm biểu đồ

Learning curves đọc từ JSONL thực tế; không điền metric giả.  
VRAM là cumulative peak tensor allocation cuối epoch — không phải VRAM tức thời hay NVML total.  
Biểu đồ Stage B bổ sung amplitude, gate weights và candidate-margin statistics.


In [ ]:
import math


DATASET_PROFILES = {
    'baby':        {'name': 'Amazon Baby',        'color': '#ff7f0e'},
    'sports':      {'name': 'Amazon Sports',      'color': '#1f77b4'},
    'electronics': {'name': 'Amazon Electronics', 'color': '#2ca02c'},
}
ARM_COLORS = {
    'B0':           '#333333',
    'mm-ss':        '#1f77b4',
    'mm-bpr':       '#ff7f0e',
    'id-ss':        '#2ca02c',
    'baseline-ss':  '#d62728',
    'baseline-bpr': '#9467bd',
}


def latest_records(records):
    """Deduplicate by epoch (keep latest attempt), sorted ascending."""
    return sorted({r['epoch']: r for r in records}.values(), key=lambda r: r['epoch'])


def observed_metric(records, metric):
    """Return (epoch, value) pairs for a metric; skip missing/nonfinite."""
    pts = []
    for r in records:
        v = {k.upper(): v for k, v in r.get('metrics', {}).items()}.get(metric.upper())
        if v is not None and math.isfinite(float(v)):
            pts.append((r['epoch'], float(v)))
    return pts


def save_figure(fig, path):
    import matplotlib.pyplot as plt
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
    finally:
        plt.close(fig)
    print(f'Figure saved: {path}')


def _iter_stage_dirs(key, stage):
    """Yield (label, directory) for a given stage (stage_a / stage_b)."""
    for seed in SEEDS:
        seed_dir = ARTIFACT_ROOT / key / f'seed{seed}'
        if stage == 'stage_a':
            teacher = seed_dir / 'teacher' / 'stage_a'
            if (teacher / 'epochs.jsonl').is_file():
                yield f'Stage A (seed {seed})', teacher
        elif stage == 'stage_b':
            # B0 = baseline with residual disabled
            b0 = seed_dir / 'teacher' / 'stage_b'
            if (b0 / 'epochs.jsonl').is_file():
                yield f'B0 / seed {seed}', b0
            for arm in ARMS:
                d = seed_dir / arm / 'stage_b'
                if (d / 'epochs.jsonl').is_file():
                    yield f'{arm} / seed {seed}', d


def plot_stage_a_curves(key):
    """Loss, validation NDCG@20, Recall@20, and epoch time for Stage A."""
    import matplotlib.pyplot as plt
    name = DATASET_PROFILES.get(key, {}).get('name', key)
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5), constrained_layout=True)
    for label, d in _iter_stage_dirs(key, 'stage_a'):
        color = '#1f77b4'
        train  = latest_records(read_jsonl(d / 'epochs.jsonl'))
        valid  = latest_records([
            r for r in read_jsonl(d / 'evaluation.jsonl')
            if r.get('mode') == 'valid' and not r.get('selected_checkpoint')])
        manifest_path = d / 'run_manifest.json'
        selected = None
        if manifest_path.is_file():
            m = json.loads(manifest_path.read_text(encoding='utf-8'))
            if m.get('status') == 'completed':
                selected = m.get('selected_epoch')
        # BPR loss
        bpr_pts = [(r['epoch'], r['bpr']) for r in train
                   if 'bpr' in r and r['bpr'] is not None and math.isfinite(float(r['bpr']))]
        if bpr_pts:
            axes[0].plot(*zip(*bpr_pts), color=color, label=label)
        # Validation metrics
        for ax, metric in ((axes[1], 'NDCG@20'), (axes[2], 'RECALL@20')):
            pts = observed_metric(valid, metric)
            if pts:
                ax.plot(*zip(*pts), color=color, label=label)
                if selected is not None and selected in dict(pts):
                    ax.scatter([selected], [dict(pts)[selected]],
                               color=color, s=50, zorder=5)
                    ax.axvline(selected, color=color, linestyle=':', alpha=.55)
        # Epoch time
        sec_pts = [(r['epoch'], float(r['train_seconds'])) for r in train
                   if 'train_seconds' in r and r['train_seconds'] is not None]
        if sec_pts:
            axes[3].plot(*zip(*sec_pts), color=color, label=label)
    titles = ('Training BPR', 'Validation NDCG@20',
              'Validation Recall@20', 'Training seconds / epoch')
    for ax, title in zip(axes, titles):
        ax.set(title=title, xlabel='Epoch')
        ax.grid(alpha=.3)
        if ax.lines:
            ax.legend(fontsize=8)
        else:
            ax.text(.5, .5, 'No data', ha='center', transform=ax.transAxes)
    fig.suptitle(f'STAIR-RAM v5 | {name} | Stage A (marker = validation-selected epoch)')
    save_figure(fig, REPORT_DIR / f'stage_a_curves_{key}.png')


def plot_stage_b_curves(key):
    """Stage B: sampled CE loss, validation NDCG@20, Recall@20, epoch time."""
    import matplotlib.pyplot as plt
    name = DATASET_PROFILES.get(key, {}).get('name', key)
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5), constrained_layout=True)
    for label, d in _iter_stage_dirs(key, 'stage_b'):
        arm = label.split(' / ')[0]
        color = ARM_COLORS.get(arm, '#7f7f7f')
        train = latest_records(read_jsonl(d / 'epochs.jsonl'))
        valid = latest_records([
            r for r in read_jsonl(d / 'evaluation.jsonl')
            if r.get('mode') == 'valid' and not r.get('selected_checkpoint')])
        manifest_path = d / 'run_manifest.json'
        selected = None
        if manifest_path.is_file():
            m = json.loads(manifest_path.read_text(encoding='utf-8'))
            if m.get('status') == 'completed':
                selected = m.get('selected_epoch')
        # Sampled CE loss (field 'loss' or 'ce_loss')
        for loss_key in ('ce_loss', 'loss', 'bpr'):
            loss_pts = [(r['epoch'], float(r[loss_key])) for r in train
                        if loss_key in r and r[loss_key] is not None
                        and math.isfinite(float(r[loss_key]))]
            if loss_pts:
                axes[0].plot(*zip(*loss_pts), color=color, label=label)
                break
        for ax, metric in ((axes[1], 'NDCG@20'), (axes[2], 'RECALL@20')):
            pts = observed_metric(valid, metric)
            if pts:
                ax.plot(*zip(*pts), color=color, label=label)
                if selected is not None and selected in dict(pts):
                    ax.scatter([selected], [dict(pts)[selected]],
                               color=color, s=50, zorder=5)
                    ax.axvline(selected, color=color, linestyle=':', alpha=.55)
        sec_pts = [(r['epoch'], float(r['train_seconds'])) for r in train
                   if 'train_seconds' in r and r['train_seconds'] is not None]
        if sec_pts:
            axes[3].plot(*zip(*sec_pts), color=color, label=label)
    titles = ('Sampled CE loss (Stage B)', 'Validation NDCG@20',
              'Validation Recall@20', 'Training seconds / epoch')
    for ax, title in zip(axes, titles):
        ax.set(title=title, xlabel='Epoch within Stage B')
        ax.grid(alpha=.3)
        if ax.lines:
            ax.legend(fontsize=8)
        else:
            ax.text(.5, .5, 'No data', ha='center', transform=ax.transAxes)
    fig.suptitle(f'STAIR-RAM v5 | {name} | Stage B (epoch 0 = exact baseline B0)')
    save_figure(fig, REPORT_DIR / f'stage_b_curves_{key}.png')


def plot_stage_b_diagnostics(key):
    """Stage B diagnostics: amplitude, gate weights, candidate margin.

    These fields are logged by main_stair4_v5 into epochs.jsonl when available.
    The plot is silently skipped if the fields are absent.
    """
    import matplotlib.pyplot as plt
    name = DATASET_PROFILES.get(key, {}).get('name', key)
    diag_fields = ['amplitude_mean', 'amplitude_std',
                   'gate_text_mean', 'gate_image_mean',
                   'candidate_margin_mean']
    has_any = False
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    for label, d in _iter_stage_dirs(key, 'stage_b'):
        arm = label.split(' / ')[0]
        color = ARM_COLORS.get(arm, '#7f7f7f')
        train = latest_records(read_jsonl(d / 'epochs.jsonl'))
        # Amplitude
        amp_pts = [(r['epoch'], float(r['amplitude_mean'])) for r in train
                   if 'amplitude_mean' in r and r['amplitude_mean'] is not None
                   and math.isfinite(float(r['amplitude_mean']))]
        if amp_pts:
            axes[0].plot(*zip(*amp_pts), color=color, label=label)
            has_any = True
        # Gates
        for gf, ls in (('gate_text_mean', '-'), ('gate_image_mean', '--')):
            gate_pts = [(r['epoch'], float(r[gf])) for r in train
                        if gf in r and r[gf] is not None and math.isfinite(float(r[gf]))]
            if gate_pts:
                axes[1].plot(*zip(*gate_pts), color=color, linestyle=ls,
                             label=f'{label} ({gf})')
                has_any = True
        # Candidate margin
        margin_pts = [(r['epoch'], float(r['candidate_margin_mean'])) for r in train
                      if 'candidate_margin_mean' in r
                      and r['candidate_margin_mean'] is not None
                      and math.isfinite(float(r['candidate_margin_mean']))]
        if margin_pts:
            axes[2].plot(*zip(*margin_pts), color=color, label=label)
            has_any = True
    if not has_any:
        print(f'{key}: no Stage B diagnostic fields in epochs.jsonl — skipping diagnostic plot.')
        import matplotlib.pyplot as plt
        plt.close(fig)
        return
    titles = ('Residual amplitude (mean)', 'Modality gate weight (mean)',
              'Candidate margin (mean)')
    for ax, title in zip(axes, titles):
        ax.set(title=title, xlabel='Epoch within Stage B')
        ax.grid(alpha=.3)
        if ax.lines:
            ax.legend(fontsize=7)
    fig.suptitle(f'STAIR-RAM v5 | {name} | Stage B diagnostics')
    save_figure(fig, REPORT_DIR / f'stage_b_diagnostics_{key}.png')


def plot_vram(key):
    """Cumulative PyTorch peak allocated memory per epoch (Stage A + Stage B).

    This is peak tensor allocation from the start of the process, sampled at
    epoch boundaries. It is not instantaneous usage or whole-device VRAM.
    """
    import matplotlib.pyplot as plt
    name = DATASET_PROFILES.get(key, {}).get('name', key)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
    for ax, stage in zip(axes, ('stage_a', 'stage_b')):
        for label, d in _iter_stage_dirs(key, stage):
            arm = label.split(' / ')[0]
            color = ARM_COLORS.get(arm, '#1f77b4')
            if stage == 'stage_a':
                color = DATASET_PROFILES.get(key, {}).get('color', '#1f77b4')
            # Skip CPU-only runs
            cmd_file = d.parent / 'command.json'
            if cmd_file.is_file():
                argv = json.loads(cmd_file.read_text(encoding='utf-8'))
                if '--device' in argv and argv[argv.index('--device') + 1] == 'cpu':
                    continue
            train = latest_records(read_jsonl(d / 'epochs.jsonl'))
            pts = [(r['epoch'], float(r['peak_allocated_mb'])) for r in train
                   if 'peak_allocated_mb' in r
                   and r['peak_allocated_mb'] is not None
                   and math.isfinite(float(r['peak_allocated_mb']))
                   and float(r['peak_allocated_mb']) > 0]
            if pts:
                ax.plot(*zip(*pts), color=color,
                        label=f'{label} (max {max(v for _, v in pts):.0f} MiB)')
        stage_label = 'Stage A (MI/FSC/BSC)' if stage == 'stage_a' else 'Stage B (residual)'
        ax.set(title=f'{name} | {stage_label} | cumulative peak allocated',
               xlabel='Epoch', ylabel='MiB')
        ax.grid(alpha=.3)
        if ax.lines:
            ax.legend(fontsize=8)
        else:
            ax.text(.5, .5, 'No CUDA memory measurements',
                    ha='center', transform=ax.transAxes)
    fig.suptitle(f'STAIR-RAM v5 | {name} | VRAM usage (not instantaneous)')
    save_figure(fig, REPORT_DIR / f'vram_{key}.png')


def _plot_after_training(key):
    """All four plot calls for a single dataset."""
    plot_stage_a_curves(key)
    plot_vram(key)
    plot_stage_b_curves(key)
    plot_stage_b_diagnostics(key)


print('Plot functions defined.')


## 7. Training — Amazon Baby


In [ ]:
train_dataset('baby', selected_only=True)


### Bộ nhớ CUDA — Baby

Cumulative peak tensor allocation từ đầu tiến trình, lấy mẫu cuối epoch.  
Không phải VRAM tức thời hoặc tổng VRAM thiết bị. Có thể chạy lại độc lập không training lại.


In [ ]:
if 'baby' in DATASETS_TO_RUN:
    plot_vram('baby')


### Learning curves Stage A — Baby

BPR loss, validation NDCG@20, Recall@20 và thời gian epoch.  
Marker đánh dấu epoch được chọn bởi validation NDCG@20.


In [ ]:
if 'baby' in DATASETS_TO_RUN:
    plot_stage_a_curves('baby')


### Learning curves Stage B — Baby

Epoch 0 là baseline chính xác (residual disabled). Từ epoch 1, residual được bật.  
Không so validation với test baseline lịch sử; không lấy maximum riêng của Recall.


In [ ]:
if 'baby' in DATASETS_TO_RUN:
    plot_stage_b_curves('baby')


### Stage B diagnostics — Baby

Amplitude residual, gate weights theo modality và candidate margin (nếu được ghi vào epochs.jsonl).


In [ ]:
if 'baby' in DATASETS_TO_RUN:
    plot_stage_b_diagnostics('baby')


## 8. Training — Amazon Sports


In [ ]:
train_dataset('sports', selected_only=True)


### Bộ nhớ CUDA — Sports


In [ ]:
if 'sports' in DATASETS_TO_RUN:
    plot_vram('sports')


### Learning curves Stage A — Sports


In [ ]:
if 'sports' in DATASETS_TO_RUN:
    plot_stage_a_curves('sports')


### Learning curves Stage B — Sports


In [ ]:
if 'sports' in DATASETS_TO_RUN:
    plot_stage_b_curves('sports')


### Stage B diagnostics — Sports


In [ ]:
if 'sports' in DATASETS_TO_RUN:
    plot_stage_b_diagnostics('sports')


## 9. Training — Amazon Electronics


In [ ]:
train_dataset('electronics', selected_only=True)


### Bộ nhớ CUDA — Electronics


In [ ]:
if 'electronics' in DATASETS_TO_RUN:
    plot_vram('electronics')


### Learning curves Stage A — Electronics


In [ ]:
if 'electronics' in DATASETS_TO_RUN:
    plot_stage_a_curves('electronics')


### Learning curves Stage B — Electronics


In [ ]:
if 'electronics' in DATASETS_TO_RUN:
    plot_stage_b_curves('electronics')


### Stage B diagnostics — Electronics


In [ ]:
if 'electronics' in DATASETS_TO_RUN:
    plot_stage_b_diagnostics('electronics')


## 10. Metrics tại checkpoint được chọn

Chỉ xuất valid/test từ record `selected_checkpoint=True` khớp `selected_epoch`.  
Delta treatment/control chỉ tính khi cùng seed, dataset, source hash và fingerprint dữ liệu.  
Một seed không đủ kết luận ý nghĩa thống kê; không dùng test để chọn arm hay seed.


In [ ]:
import pandas as pd

METRICS = ['RECALL@10', 'RECALL@20', 'NDCG@10', 'NDCG@20']


def collect_results():
    """Scan all completed Stage B runs and build a summary DataFrame."""
    rows = []
    for seed in SEEDS:
        for key in DATASETS_TO_RUN:
            seed_dir = ARTIFACT_ROOT / key / f'seed{seed}'
            # Include B0 (teacher / stage_b) and all arms
            arm_dirs = [('B0', seed_dir / 'teacher' / 'stage_b')]
            arm_dirs += [(arm, seed_dir / arm / 'stage_b') for arm in ARMS]
            for arm, d in arm_dirs:
                manifest_path = d / 'run_manifest.json'
                if not manifest_path.is_file():
                    continue
                m = json.loads(manifest_path.read_text(encoding='utf-8'))
                row = {
                    'dataset': key, 'arm': arm, 'seed': seed,
                    'status': m.get('status', 'unknown'),
                    'selected_epoch': m.get('selected_epoch'),
                    'run': str(d),
                }
                # Timing
                epoch_records = latest_records(read_jsonl(d / 'epochs.jsonl'))
                sec = [r['train_seconds'] for r in epoch_records
                       if 'train_seconds' in r and r['train_seconds'] is not None
                       and r['epoch'] > 0]  # skip epoch 0 (baseline eval)
                row['median_train_seconds'] = (
                    float(pd.Series(sec, dtype=float).median()) if sec else float('nan'))
                row['peak_allocated_mb'] = (
                    max((float(r['peak_allocated_mb']) for r in epoch_records
                         if 'peak_allocated_mb' in r
                         and r['peak_allocated_mb'] is not None
                         and float(r['peak_allocated_mb']) > 0), default=float('nan')))
                # Metrics at selected checkpoint
                for mode in ('valid', 'test'):
                    for metric in METRICS:
                        row[f'{mode}_{metric}'] = float('nan')
                if m.get('status') == 'completed':
                    sel = m.get('selected_epoch')
                    for r in read_jsonl(d / 'evaluation.jsonl'):
                        if (r.get('mode') in ('valid', 'test')
                                and r.get('selected_checkpoint')
                                and r.get('epoch') == sel):
                            for metric in METRICS:
                                v = r.get('metrics', {}).get(metric)
                                if v is not None:
                                    row[f"{r['mode']}_{metric}"] = float(v)
                rows.append(row)

    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame

    # Compute delta vs B0 (same dataset/seed)
    for metric in METRICS:
        frame[f'delta_valid_{metric}_pct'] = float('nan')
        if not frame[[c for c in frame.columns if c.startswith('test_')]].empty:
            frame[f'delta_test_{metric}_pct'] = float('nan')
    for idx, row in frame.iterrows():
        if row.get('arm') == 'B0':
            continue
        controls = frame[(frame.dataset == row.dataset)
                         & (frame.seed == row.seed)
                         & (frame.arm == 'B0')
                         & (frame.status == 'completed')]
        if len(controls) != 1 or row.status != 'completed':
            continue
        b0 = controls.iloc[0]
        for mode in ('valid', 'test'):
            for metric in METRICS:
                col = f'{mode}_{metric}'
                delta_col = f'delta_{mode}_{metric}_pct'
                if delta_col not in frame.columns:
                    continue
                val, base = row.get(col, float('nan')), b0.get(col, float('nan'))
                if (pd.notna(val) and pd.notna(base)
                        and math.isfinite(float(val)) and math.isfinite(float(base))
                        and base > 0):
                    frame.loc[idx, delta_col] = 100.0 * (float(val) / float(base) - 1)
    return frame


In [ ]:
RESULTS = collect_results()
if RESULTS.empty:
    print('No completed training runs found.')
else:
    RESULTS.to_csv(REPORT_DIR / 'all_runs.csv', index=False)
    display(RESULTS[['dataset', 'arm', 'seed', 'status', 'selected_epoch',
                      'median_train_seconds', 'peak_allocated_mb']
                     + [f'valid_{m}' for m in METRICS]
                     + [f'test_{m}' for m in METRICS]])
    # Seed summary
    completed = RESULTS[RESULTS.status == 'completed']
    if not completed.empty:
        agg = completed.groupby(['dataset', 'arm'])[
            [f'valid_{m}' for m in METRICS]].agg(['mean', 'std', 'count'])
        agg.to_csv(REPORT_DIR / 'seed_summary.csv')
        display(agg)
    # LaTeX table
    cols = ['dataset', 'arm', 'seed', 'selected_epoch'] + [f'valid_{m}' for m in METRICS]
    (REPORT_DIR / 'selected_metrics.tex').write_text(
        RESULTS[cols].to_latex(
            index=False, na_rep='--',
            float_format=lambda x: f'{x:.5f}',
            caption='Validation metrics at the selected checkpoint. '
                    'B0 = baseline with residual disabled. '
                    'Pilot mode: test is undisclosed.',
            escape=True),
        encoding='utf-8')
    print('Reports saved to:', REPORT_DIR)


## 11. Paired analysis (confirm mode only)

Pilot không có test nên analyzer báo thiếu test là đúng.  
Confirmation chỉ dùng test tại checkpoint được validation chọn.  
Không so metric test với validation hoặc lấy riêng từng metric ở epoch tốt nhất.


In [ ]:
if MODE == 'confirm':
    subprocess.run(
        [sys.executable, 'scripts/analyze_stair4_v5.py', str(ARTIFACT_ROOT),
         '--output', str(REPORT_DIR),
         '--treatment', 'mm-ss', '--control', 'B0',
         '--seeds', *map(str, SEEDS)],
        cwd=REPO, env=ENV, check=True)
    analysis_file = REPORT_DIR / 'paired_analysis.json'
    if analysis_file.is_file():
        print(analysis_file.read_text(encoding='utf-8'))
else:
    print('Pilot mode: test remains undisclosed. Use MODE="confirm" after locking configuration.')
print('Artifacts:', ARTIFACT_ROOT)


## 12. Đóng gói để tải về

Archive bao gồm: logs, JSONL, reports, source snapshot, runtime.json, pip-freeze.txt.  
Không đóng gói dataset và feature file lớn; không đóng gói thư mục preflight tạm.


In [ ]:
if MAKE_ARCHIVE:
    archive = ARTIFACT_ROOT.parent / (EXPERIMENT_ID + '_artifacts.zip')
    exclude_dirs = {'preflight-tmp', 'preflight'}
    exclude_suffixes = {'.pkl', '.pt'}  # Skip large model/feature binary files
    with zipfile.ZipFile(archive, 'w',
                         compression=zipfile.ZIP_DEFLATED, compresslevel=1) as bundle:
        for path in sorted(ARTIFACT_ROOT.rglob('*')):
            if not path.is_file():
                continue
            relative = path.relative_to(ARTIFACT_ROOT)
            if relative.parts[0] in exclude_dirs:
                continue
            if path.suffix in exclude_suffixes:
                continue
            bundle.write(path, Path(EXPERIMENT_ID) / relative)
    size_mib = archive.stat().st_size / 2**20
    print(f'Archive: {archive} ({size_mib:.1f} MiB)')
else:
    print('Artifacts retained at:', ARTIFACT_ROOT)


## 13. Resume và chạy riêng từng dataset

Để resume Stage B bị gián đoạn:

    resume_stage_b('sports', 1, 'mm-ss')

Để chạy riêng một dataset ngoài DATASETS_TO_RUN:

    train_dataset('electronics')

Resume không thay đổi budget hay config. Lần chạy hoàn tất không cần resume.  
Muốn thêm control, thêm arm vào ARMS và chạy lại training cells — teacher không train lại.


## 14. Xử lý lỗi và đọc kết quả

- Traceback ở plotting không có nghĩa training bị hỏng. Kiểm tra `run_manifest.json` có `status=completed`.
- Biểu đồ Stage B diagnostics bị bỏ qua nếu không tìm thấy field `amplitude_mean` trong `epochs.jsonl` — không phải lỗi.
- TorchData deprecation / CSR beta warning không phải lỗi; không dùng warning filter.
- Thời gian, VRAM và mức tăng metric phải đo trên Kaggle thực tế — không có giá trị mô phỏng.
- Để chạy lại plot mà không training lại: chạy lại các cell định nghĩa hàm, sau đó gọi trực tiếp hàm plot.
